In [2]:

import os
import re
import sys
import argparse
import textwrap
from typing import List, Optional

from dotenv import load_dotenv

load_dotenv()

try:
    import pyodbc
except ImportError:
    print("ERROR: pyodbc not installed. Install with: pip install pyodbc", file=sys.stderr)
    sys.exit(1)

In [3]:
VIEW_QUERY = """
SELECT
    DB_NAME() AS database_name,
    s.name AS schema_name,
    'VIEW' AS object_type,
    v.name AS object_name,
    sm.definition AS definition
FROM sys.views AS v
JOIN sys.schemas AS s
    ON v.schema_id = s.schema_id
JOIN sys.sql_modules AS sm
    ON v.object_id = sm.object_id
ORDER BY s.name, v.name;
"""

PROC_QUERY = """
SELECT
    DB_NAME() AS database_name,
    s.name AS schema_name,
    'PROCEDURE' AS object_type,
    p.name AS object_name,
    sm.definition AS definition
FROM sys.procedures AS p
JOIN sys.schemas AS s
    ON p.schema_id = s.schema_id
JOIN sys.sql_modules AS sm
    ON p.object_id = sm.object_id
ORDER BY s.name, p.name;
"""

FN_QUERY = """
SELECT
    DB_NAME() AS database_name,
    s.name AS schema_name,
    'FUNCTION' AS object_type,
    o.name AS object_name,
    sm.definition AS definition
FROM sys.objects AS o
JOIN sys.schemas AS s
    ON o.schema_id = s.schema_id
JOIN sys.sql_modules AS sm
    ON o.object_id = sm.object_id
WHERE o.type IN ('FN', 'TF', 'IF')  -- FN = Scalar, TF = Table-valued, IF = Inline Table-valued
ORDER BY s.name, o.name;
"""

def sanitize_segment(name: str) -> str:
    """
    Sanitize a path segment for use as a folder or file name:
    - Replace spaces with underscores
    - Allow letters, digits, underscore, hyphen, and dot
    - Replace any other character with underscore
    """
    if not isinstance(name, str):
        name = str(name)
    name = name.strip().replace(' ', '_')
    return re.sub(r'[^A-Za-z0-9._-]', '_', name)

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def get_connection(server: str, database: Optional[str], user: Optional[str], password: Optional[str],
                   driver: str, trusted: bool = False) -> pyodbc.Connection:
    """
    Build and return a pyodbc connection to the given database.
    """

    parts = [
        "DRIVER={ODBC Driver 17 for SQL Server}",
        f"SERVER={server}",
        f"Encrypt=yes",
        f"TrustServerCertificate=yes",
        f"UID={user}",
        f"PWD={password}"
    ]

    if database:
        parts.append(f"DATABASE={database}")

    # if trusted:
    #     parts.append("Trusted_Connection=yes")
    # else:
    #     if not user or not password:
    #         raise ValueError("Username and password are required when not using --trusted")
    #     parts.append(f"UID=DEVADMIN")
    #     parts.append(f"PWD=Devdlkh@2023.")

    conn_str = ";".join(parts)
    print(conn_str)
    return pyodbc.connect(conn_str)

def fetch_rows(cursor: pyodbc.Cursor, sql: str) -> List[pyodbc.Row]:
    cursor.execute(sql)
    return cursor.fetchall()


def write_definition(root_out: str, database_name: str, schema_name: str,
                     object_type: str, object_name: str, definition: str) -> str:
    """
    Write the DDL definition to the appropriate file path and return the file path.
    """
    db = sanitize_segment(database_name)
    schema = sanitize_segment(schema_name)
    objtype = sanitize_segment(object_type)
    objname = sanitize_segment(object_name)

    folder = os.path.join(root_out, db, schema, objtype)
    ensure_dir(folder)

    file_path = os.path.join(folder, f"{objname}.sql")

    text = definition or ""

    # Normalize all line endings to \n
    text = text.replace('\r\n', '\n').replace('\r', '\n')

    # Remove excessive blank lines (collapse 3+ into 2)
    import re
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Ensure single trailing newline
    text = text.rstrip() + '\n'

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(text)

    return file_path



def export_database(server: str, db_name: str, user: Optional[str], password: Optional[str],
                    driver: str, root_out: str, trusted: bool) -> None:
    """
    Export views and procedures from the given database.
    """
    print(f"==> Exporting from database: {db_name}")
    conn = None
    try:
        print(driver)
        conn = get_connection(server=server, database=db_name, user=user,
                              password=password, driver=driver, trusted=trusted)
        cursor = conn.cursor()

        total_written = 0

        for label, sql in (("views", VIEW_QUERY), ("procedures", PROC_QUERY), ('functions', FN_QUERY)):
            rows = fetch_rows(cursor, sql)
            print(f"   - Found {len(rows)} {label}")
            for r in rows:
                db = r.database_name
                schema = r.schema_name
                objtype = r.object_type
                objname = r.object_name
                definition = r.definition

                # Skip encrypted or NULL definitions
                if definition is None:
                    print(f"     ! Skipping {schema}.{objname} ({objtype}) - definition is NULL (possibly encrypted)")
                    continue

                out_path = write_definition(root_out, db, schema, objtype, objname, definition)
                total_written += 1

        print(f"   ✓ Wrote {total_written} files for {db_name}")

    except pyodbc.Error as e:
        print(f"   ✗ ODBC error exporting {db_name}: {e}", file=sys.stderr)
    except Exception as e:
        print(f"   ✗ Unexpected error exporting {db_name}: {e}", file=sys.stderr)
    finally:
        try:
            if conn is not None:
                conn.close()
        except Exception:
            pass

def main():

    SERVER = os.getenv("SQLSERVER_URL")
    USER = os.getenv("SQLSERVER_USER")
    PWD = os.getenv("SQLSERVER_PWD")
    DRIVER = "ODBC Driver 18 for SQL Server"
    OUTPUT_FOLDER = '.'
    DATABASES = ['DLKHSTAGE', 'DLKHDIM', 'DLKHFACT']


    # Ensure output root directory exists
    ensure_dir('.')

    for db in DATABASES:
        export_database(
            server=SERVER,
            db_name=db,
            user=USER,
            password=PWD,
            driver=DRIVER,
            root_out=OUTPUT_FOLDER,
            trusted=True
        )

    print("All done.")

if __name__ == "__main__":
    main()



==> Exporting from database: DLKHSTAGE
ODBC Driver 18 for SQL Server
DRIVER={ODBC Driver 17 for SQL Server};SERVER=10.116.17.52;Encrypt=yes;TrustServerCertificate=yes;UID=sa;PWD=Db@dm!nD1K#*6;DATABASE=DLKHSTAGE
   - Found 28 views
   - Found 205 procedures
   - Found 16 functions
   ✓ Wrote 249 files for DLKHSTAGE
==> Exporting from database: DLKHDIM
ODBC Driver 18 for SQL Server
DRIVER={ODBC Driver 17 for SQL Server};SERVER=10.116.17.52;Encrypt=yes;TrustServerCertificate=yes;UID=sa;PWD=Db@dm!nD1K#*6;DATABASE=DLKHDIM
   - Found 0 views
   - Found 78 procedures
   - Found 8 functions
   ✓ Wrote 86 files for DLKHDIM
==> Exporting from database: DLKHFACT
ODBC Driver 18 for SQL Server
DRIVER={ODBC Driver 17 for SQL Server};SERVER=10.116.17.52;Encrypt=yes;TrustServerCertificate=yes;UID=sa;PWD=Db@dm!nD1K#*6;DATABASE=DLKHFACT
   - Found 0 views
   - Found 29 procedures
   - Found 1 functions
   ✓ Wrote 30 files for DLKHFACT
All done.
